# 01 — Exploratory analysis

Explore the data before trusting any model, using the same modules the
pipeline uses so the two cannot drift apart.

**Prerequisite:** run `python -m scripts.run_data` first. This notebook reads
the cached tables rather than rebuilding them, which takes seconds instead of
minutes.

In [ ]:
import sys
from pathlib import Path

# Let a notebook living in notebooks/ see the src package.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.config import PROCESSED_DIR, load_config
from src.utils import load_table

pd.set_option("display.max_columns", 60)
plt.rcParams["figure.figsize"] = (11, 4)

cfg = load_config()
TOP_N = cfg["features"]["top_n"]
cfg["data"]

## 1. Load the prepared tables

In [ ]:
dataset = load_table(PROCESSED_DIR / "dataset.parquet")
top10 = load_table(PROCESSED_DIR / "top10_daily.parquet")
concentration = load_table(PROCESSED_DIR / "concentration.parquet")

for frame in (dataset, top10, concentration):
    frame["date"] = pd.to_datetime(frame["date"])

print(f"dataset      {dataset.shape[0]:,} rows x {dataset.shape[1]} cols")
print(f"window       {dataset['date'].min():%Y-%m-%d} to {dataset['date'].max():%Y-%m-%d}")
dataset.head()

In [ ]:
# Where are the nulls? Expect a warm-up block at the start (rolling windows not
# yet full) and one row at the end (no t+1 target). Neither may be imputed.
nulls = dataset.isna().sum()
nulls[nulls > 0]

## 2. Who is in the block?

The columns are rank slots, not tickers. `x1` is whatever company is largest on
that date, which is what makes the design survive constituents dropping out.

In [ ]:
name_cols = [f"name_{i}" for i in range(1, TOP_N + 1)]

for stamp in ["2016-01-05", "2020-01-06", "2025-12-31"]:
    row = top10[top10["date"] == stamp]
    if len(row):
        tickers = ", ".join(str(row.iloc[0][c]) for c in name_cols)
        print(f"{stamp}  {tickers}")

In [ ]:
# How much churn is there really?
long = top10.melt(id_vars="date", value_vars=name_cols,
                  var_name="slot", value_name="ticker").dropna(subset=["ticker"])

tenure = (long.groupby("ticker")["date"]
          .agg(first_seen="min", last_seen="max", days="count")
          .sort_values("days", ascending=False))

print(f"{len(tenure)} distinct companies passed through a block {TOP_N} wide")
print(f"{(tenure['days'] == len(top10)).sum()} were present every single day")
tenure

## 3. Concentration

Computed inside the block only, so it stays within the scope of the research
question. A ratio against all 500 would import information about the other 490.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)

axes[0].plot(concentration["date"], concentration["top_n_total"], lw=1.2)
axes[0].set_title(f"Combined market cap of the top {TOP_N} (USD bn)")

axes[1].plot(concentration["date"], concentration["share_1"], label="share_1", lw=1.2)
axes[1].plot(concentration["date"], concentration["hhi"], label="hhi", lw=1.2)
axes[1].set_title("Concentration inside the block")
axes[1].legend()

plt.tight_layout()
plt.show()

## 4. The drift — the central finding

`index / block market cap` is the coefficient any model has to learn. If it were
stable, a model fitted on the past would transfer to the future. It is not.

In [ ]:
from src.features.build_features import mcap_feature_columns

xs = mcap_feature_columns(TOP_N)
frame = dataset.dropna(subset=["target"]).copy()
frame["S"] = frame[xs].sum(axis=1)
frame["ratio"] = frame["target"] / frame["S"]

yearly = frame.groupby(frame["date"].dt.year)["ratio"].mean()
print(yearly.round(4).to_string())
print(f"\ndrift: {yearly.iloc[0] / yearly.iloc[-1]:.2f}x")

plt.plot(frame["date"], frame["ratio"], lw=1)
plt.title("Index level divided by top-block market cap")
plt.show()

## 5. Explanatory power, year by year

Fit inside a year and score inside the same year. This asks how much of the
index the constituents account for — not how well the relationship projects
forward, which is a different question with a different answer.

In [ ]:
from src.models.explanatory import coefficient_drift, within_period_fit

periods = within_period_fit(dataset, xs)
display(periods[["year", "n", "r2", "mape", "index_over_block"]].round(4))

stats = coefficient_drift(periods)
for key, value in stats.items():
    print(f"{key:24s} {value:10.4f}")

## 6. Check the expanding-window folds

Confirm the split behaves: training always precedes validation, the window
grows, and there is no overlap.

In [ ]:
from src.models.split import expanding_year_folds

clean = dataset.dropna(subset=xs + ["target"]).reset_index(drop=True)
folds = expanding_year_folds(clean["date"], cfg)

rows = []
for fold in folds:
    train_max = clean["target"].iloc[fold.train_idx].max()
    valid = clean["target"].iloc[fold.valid_idx]
    rows.append({
        "fold": fold.name,
        "n_train": len(fold.train_idx),
        "n_valid": len(fold.valid_idx),
        "overlap": len(set(fold.train_idx) & set(fold.valid_idx)),
        "train_max": round(train_max),
        "valid_max": round(valid.max()),
        "pct_above_train_max": round((valid > train_max).mean() * 100, 1),
    })

pd.DataFrame(rows)

The last column is why tree models struggle here. They cannot predict above the
highest target seen in training, so any fold with a high percentage is asking
for a number they cannot produce.

**2022 and 2023 are the controls** — the two validation years that stay inside
the range already seen. They do not behave alike: 2022 is the best fold in the
study while 2023 is one of the worst, which is the first sign that the ceiling
is not the only thing going wrong. The other half is the drift from section 4,
and the Model Comparison page of the dashboard separates the two.

## 7. Leakage check

`target(t)` should be the index close at `t+1`, and `baseline_naive(t)` the
close at `t`. If the design holds, tomorrow's baseline equals today's target.

In [ ]:
horizon = cfg["target"]["horizon"]
aligned = dataset.dropna(subset=["target", "baseline_naive"]).reset_index(drop=True)

matches = np.allclose(
    aligned["target"].to_numpy()[:-horizon],
    aligned["baseline_naive"].to_numpy()[horizon:],
)
print(f"target(t) == baseline(t+{horizon}): {matches}")

# Correlation of each feature with the target. Anything suspiciously close to 1
# among the trend columns would suggest the target leaked into a feature.
feature_cols = [c for c in dataset.columns
                if c not in {"date", "target", "baseline_naive"}]
dataset[feature_cols].corrwith(dataset["target"]).sort_values(key=abs, ascending=False)

The `x` columns correlating near 1.0 with the target is expected and not
leakage: both are market values rising together over a decade. The trend and
macro columns are the ones worth checking, and they should be far from 1.

## Next steps

```bash
python -m scripts.run_train         # expanding-window ablation A / B / C
python -m scripts.run_experiments   # N sweep, explanatory power, importance
streamlit run app/Home.py           # dashboard
```